# Traffic Speed Visualization
Visualizes road segment average speeds using MapboxGL.

In [91]:
# 1. Install Dependencies (if needed)
# !pip install requests pandas geopandas mapboxgl shapely


In [92]:
# 2. Setup
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from mapboxgl.viz import LinestringViz
from mapboxgl.utils import create_color_stops

load_dotenv()

MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN", "your_mapbox_token_here")
BASE_URL = "http://localhost:8000"

In [93]:
# 3. One-time bootstrap (if API/data are not ready)
# Run these in a terminal from repo root:
# make up
# make migrate
# make ingest


In [94]:
# 4. Request Aggregated Data
params = {
    "day": "Monday",
    "period": "AM Peak",
    "limit": 1000,
}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
geojson_data = response.json()
print(f"Fetched {len(geojson_data)} road segments")


Fetched 1000 road segments


In [95]:
# 5. Visualize in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "road_name": f["road_name"],
            "link_id": f["link_id"],
            "average_speed": f["average_speed"],
            "length_m": f["length_m"],
        },
    }
    for f in geojson_data
]

viz = LinestringViz(
    {"type": "FeatureCollection", "features": features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="RdYlGn"),
    center=(-81.6557, 30.3322),
    zoom=11,
    line_width_default=1.5,
    opacity=0.8,
)
viz.show()

In [96]:
# 6. Tabular Summary — 10 slowest road segments
df = pd.DataFrame([
    {
        "link_id": f["link_id"],
        "avg_speed": f["average_speed"],
        "road_name": f["road_name"],
        "length_m": f["length_m"],
    }
    for f in geojson_data
])
df.sort_values("avg_speed").head(10)

,link_id,avg_speed,road_name,length_m
536,1021821186,0.62,Swelo Rd,16.9
541,1021905649,1.24,Regency Court Mall,20.6
795,1033382080,1.24,None,15.5
673,1032710453,1.24,Mt Herman St,120.5
719,1032971696,1.24,None,87.9
979,1049745903,1.55,Bartram Office Park,83.4
79,1006497018,1.55,Irongate Dr,76.5
740,1032980614,1.71,Saints Rd,76.1
744,1032983340,1.86,Kingsbury St,22.2
916,1046257776,1.86,Grasse St,191.5


## Spatial Filter
Query road segments within a custom bounding box. Adjust the coordinates and re-run.

In [97]:
# 7. Spatial Filter — adjust these parameters and re-run
BBOX_DAY    = "Monday"
BBOX_PERIOD = "AM Peak"
BBOX_MIN_LON = -81.70
BBOX_MIN_LAT =  30.20
BBOX_MAX_LON = -81.60
BBOX_MAX_LAT =  30.35

resp = requests.post(
    f"{BASE_URL}/aggregates/spatial_filter/",
    json={
        "day": BBOX_DAY,
        "period": BBOX_PERIOD,
        "bbox": [BBOX_MIN_LON, BBOX_MIN_LAT, BBOX_MAX_LON, BBOX_MAX_LAT],
    },
    params={"limit": 1000},
)
resp.raise_for_status()
bbox_data = resp.json()
print(f"Found {len(bbox_data)} road segments in bbox")

Found 1000 road segments in bbox


In [98]:
# 8. Visualize spatial filter results
bbox_features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "road_name": f["road_name"],
            "link_id": f["link_id"],
            "average_speed": f["average_speed"],
            "length_m": f["length_m"],
        },
    }
    for f in bbox_data
]

center_lon = (BBOX_MIN_LON + BBOX_MAX_LON) / 2
center_lat = (BBOX_MIN_LAT + BBOX_MAX_LAT) / 2

bbox_viz = LinestringViz(
    {"type": "FeatureCollection", "features": bbox_features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="RdYlGn"),
    center=(center_lon, center_lat),
    zoom=12,
    line_width_default=2.0,
    opacity=0.9,
)
bbox_viz.show()

/home/phyllis/.pyenv/versions/3.12.4/lib/python3.12/site-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
